# Create Rufford Foundation Awards

Creates awards for the **Rufford Foundation** (UK charity 1117270; OpenAlex F4320309818), which funds
nature-conservation projects in developing countries through the **Rufford Small Grants** programme
(first, second and booster grants, mostly field research and monitoring led by early-career
conservationists), from the foundation's own **Project Directory**
(https://www.rufford.org/projects/, static Django HTML; one page per grant at
`/projects/<grantee-slug>/<project-slug>/`). There is no bulk export, API, 360Giving file or Europe
PMC (GRIST) feed for Rufford (checked 2026-10-01).

**Coverage:** every project in the directory: **6,809 grants, 1998-2026**. All kept (scope rule:
Rufford's small grants are conservation research / practice; no filter).

**Fields:** `display_name` = project title; `description` = the page's summary + body text;
`lead_investigator` = the grantee named on the page (split with the canonical `split_name`;
affiliation not published, NULL); `start_date` = the project date shown on the page; place, ISO
country code, continent and species/theme categories stay in the raw table.
`amount` NULL: Rufford does not publish per-grant amounts (Rufford Small Grants are up to GBP 6,000 /
12,000 / 15,000 by round); §6.7 amount check waived for that reason.

**`funder_award_id` (§2.1.1):** Rufford numbers grants `NNNNN-S` (application number + grant
sequence: `-1` first grant, `-2` second, `-B` booster, ...; e.g. `33202-1`, `23387-B`) and older
grants `N.MM.YY` (e.g. `130.07.04`); that is the form grantees cite (634 distinct `NNNNN-S`
references among Rufford's priority-0 citation rows). The pages do not print it as text, but every
uploaded report PDF and most project photos are filed under it, so the script takes it from the
report file names, then the image folders, and keeps it only when no other page claims it:
**5,839 (85.8%) grants carry the reference**; the other 970 (415 grants of 2024-2026 with no report filed yet,
~460 pre-2011 grants, and 105 whose reference another page also claims) ship a
stable synthetic key `RUFFORD-<grantee-slug>/<project-slug>` (the page path).

**Prerequisites:** run `scripts/local/rufford_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/rufford/rufford_projects.parquet` (§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320309818`, Rufford Foundation, GB;
ror 02bxrrf91 and doi 10.13039/100007463 read from `openalex.funders.funders`.

**Priority:** `574` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rufford_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rufford/rufford_projects.parquet`;

In [ ]:
%sql
SELECT COALESCE(regexp_replace(project_ref_source, ':.*', ''), 'none') as ref_source, COUNT(*) as n
FROM openalex.awards.rufford_raw
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.rufford_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.rufford_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'Rufford F4320309818 must have exactly one openalex.funders.funders row') as funder_ok,
       first(display_name) as display_name, first(ror_id) as ror_id, first(doi) as doi, first(country_code) as country_code
FROM openalex.funders.funders
WHERE funder_id = 4320309818;

## Step 2: Create Rufford Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rufford_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320309818  -- Rufford Foundation
),

lead AS (
    SELECT
        funder_award_id,
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', TRIM(lead_family_name),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
        ) END AS lead_investigator
    FROM openalex.awards.rufford_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    'Rufford Small Grants for Nature Conservation' as funder_scheme,
    'rufford_project_directory' as provenance,
    TRY_TO_DATE(g.award_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.award_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN l.lead_investigator IS NOT NULL THEN array(l.lead_investigator) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.rufford_raw g
JOIN lead l ON l.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check (fails the run BEFORE the insert): every key is a Rufford reference
-- (NNNNN-S or N.MM.YY) or the RUFFORD-<page path> synthetic key; ids unique; no rows lost.
SELECT
    assert_true(COUNT_IF(NOT (funder_award_id RLIKE '^[0-9]{4,6}-([A-Z]|[0-9]{1,2})$'
                              OR funder_award_id RLIKE '^[0-9]{1,3}[.][0-9]{1,2}[.][0-9]{2}$'
                              OR funder_award_id RLIKE '^RUFFORD-[^/]+/[^/]+$')) = 0,
                'funder_award_id is not a Rufford reference or page-path key') as shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') as ids_unique,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.rufford_raw), 'rows dropped in transform') as all_rows,
    COUNT(*) as total
FROM openalex.awards.rufford_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rufford_project_directory' AND priority = 574;

-- Insert into openalex_awards_raw with priority.
-- Priority 574: direct-from-funder ingest of the Rufford Project Directory.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 574 outranks the priority-0
-- acknowledgement shells that share the NNNNN-S references.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    574 as priority
FROM openalex.awards.rufford_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.rufford_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, start_date,
       lead_investigator.given_name, lead_investigator.family_name
FROM openalex.awards.rufford_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt,
       COUNT_IF(NOT funder_award_id LIKE 'RUFFORD-%') as with_reference
FROM openalex.awards.rufford_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.4a frequency check: PI top-20 must be a real long-tail (repeat grantees get 2nd/booster grants).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.rufford_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.rufford_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.3 coverage (amount NULL by design: Rufford does not publish per-grant amounts).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name
FROM openalex.awards.rufford_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs (same funder): NNNNN-S references upgrade these.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(DISTINCT c.id) as distinct_awards
FROM openalex.awards.openalex_awards_raw c
JOIN openalex.awards.rufford_awards t ON t.id = c.id
WHERE c.funder_id = 4320309818 AND c.priority = 0
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rufford_project_directory'
GROUP BY provenance, priority;